# Six-Supergroup Retrieval Experiment

## Why redefine the retrieval groups?

The first comparison used the original 20 Newsgroups labels as retrieval relevance groups and showed weak results for a single PLSA model. That raised a question about the evaluation itself: do the 20 labels describe document relationships in a way that makes sense for retrieval? Many labels are closely related subtopics—such as different kinds of computer hardware or neighboring political and religious discussions—while a user looking for one broad subject may reasonably consider documents from several labels relevant. Treating every original label as a completely separate target can split related material across many narrow bins. An unsupervised model may represent those related documents together, while a 20-label score counts that as an error.

We therefore proposed a broader, human-defined grouping: combine all 20 source categories into six topical supergroups, then ask whether retrieval looks different when relevance is judged at that broader level. This is an evaluation hypothesis, not a way of teaching PLSA the labels. The model still learns word co-occurrence from document text only; the supergroup labels define the train/test stratification and which held-out documents count as relevant to each query.

We use six groups because they bring together the related source categories while retaining distinct broad areas—computing, technology/engineering, health/space, recreation/consumer exchange, religion/worldview, and politics/international affairs. K=6 is a separate modelling choice tested alongside this grouping: it asks how a six-topic PLSA behaves under six-group evaluation, but does not force a one-to-one topic-to-group mapping. The results will show whether this particular grouping helps the comparison; they cannot establish that these are the only or objectively correct clusters.


All shared implementations are imported from `src/`. Run this notebook from the repository root. Old execution outputs were cleared during refactoring. The final cell writes tables and metadata for `Result.py`; experiment-specific loops remain here.

## Environment

The next cell contains the optional dependency-install command for notebook environments where these packages are not already available.


In [ ]:
# Install from the repository root: pip install -r requirements.txt
# Then: python -m spacy download en_core_web_sm


## 1. Libraries and shared settings

The imports cover sparse count matrices and PLSA, TF-IDF and truncated-SVD/LSI baselines, the six-group data split, retrieval scoring, plots, and the topic-alignment diagnostics used later.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42


from src.config import EXPERIMENTS, SUPERGROUPS, CATEGORY_QUERIES, SUPERGROUP_QUERIES, QUERY_BANK
from src.dataset import load_corpus
from src.preprocessing import preprocess_texts, split_train_test, split_train_val_test, vectorize_counts
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import fit_baselines, plsa_query_log_likelihood, background_word_probabilities
from src.metrics import metrics_at_k
from src.experiments import make_mixed_document, average_rank_score
from src.results import export_results
EXPERIMENT = 'Six_Category_Experiment'
CONFIG = EXPERIMENTS[EXPERIMENT]


## 2. Load the corpus and define the six supergroups

Before fitting any model or inspecting retrieval scores, explicitly map each of the 20 original categories into one of six broad subject families. This grouping is the proposed answer to the evaluation concern above: related subtopics are treated as relevant to the same broad query, while distinct subject areas remain separate.

The output lists each source category and its document count. The corpus is not downsampled to make the six groups equal in size: all 18,846 documents remain available, and the split preserves the natural group proportions. The mapping is checked so every original category appears exactly once. Labels are used to create the evaluation groups, stratify the split, and score retrieval; PLSA's EM fitting uses only word counts.


In [ ]:
# Shared loader; corpus selection remains an explicit experimental choice.
corpus = load_corpus(CONFIG["corpus"], seed=SEED)
documents, labels, target_names = corpus.documents, corpus.labels, corpus.target_names
original_labels = corpus.original_labels
source_category_names = corpus.source_category_names
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
print("Corpus:", CONFIG["corpus"], "Documents:", len(documents))
for i, label in enumerate(target_names):
    print(label, int((labels == i).sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### What the grouping output shows

All 20 source categories were assigned exactly once. The six groups have different natural sizes: Computing and Recreation/consumer exchange are largest, while Digital technology/engineering and Health/space contain two original categories each. These counts describe the corpus and are not forced to equality. Keeping the original documents avoids discarding examples simply to create balanced groups.

This mapping defines the relevance judgments for the experiment. It is a deliberate, interpretable hypothesis about which categories should count as related in retrieval; it is not a clustering discovered by PLSA and is not supplied to the model during training.


## 3. Hold out test documents and build the text representation

Use a stratified 80/20 split over the six supergroups, retaining the full training partition rather than trimming it to the smallest group's size. Fit the vocabulary on training documents only, then transform the held-out test documents with that vocabulary. The same split and preprocessed text feed PLSA and the TF-IDF/LSI comparisons.


In [ ]:
# Split raw text first. Fit vocabulary only on training text.
docs_train, docs_test, y_train, y_test, original_y_train, original_y_test = split_train_test(documents, labels, original_labels, test_size=CONFIG["test_size"], seed=SEED)
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer, X_train, X_test = vectorize_counts(docs_train_processed, docs_test_processed)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Train/test count matrices:", X_train.shape, X_test.shape)
print("Non-empty test texts:", int(nonempty_test_mask.sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Split and preprocessing output

The split contains 15,076 training documents and 3,770 held-out test documents, out of all 18,846 corpus documents. The feature matrices have 3,000 columns, so every method uses the same training-derived vocabulary. Preprocessing leaves 446 training and 121 test documents with no retained terms; these have no usable vector for retrieval, so the later ranking comparison uses the 3,649 non-empty test documents.

## 4. How PLSA represents documents

PLSA models each document as a mixture of latent topics. For a document \(d\), word \(w\), and topic \(z\),

\[P(w\mid d)=\sum_z P(w\mid z)P(z\mid d).\]

The model estimates a topic-word distribution \(P(w\mid z)\) and a document-topic distribution \(P(z\mid d)\) from word counts. During fitting, EM repeatedly assigns observed word counts probabilistically to topics and updates these distributions. The category labels do not enter those updates.


In [ ]:
# The single EM and fold-in implementation lives in src/plsa_model.py.
from src.plsa_model import PLSA


## 5. Fit the primary PLSA model

The primary PLSA uses K=6 to examine whether six latent topics work well under the six-supergroup relevance definition. The labels do not pin topics to groups. EM can settle at different local optima, so five initializations are fitted on the entire 80% training partition. The selected model is the run with the highest training log-likelihood; held-out retrieval scores are not used to select it. The settings match the baseline family: up to 100 iterations, tolerance 1e-4, beta=1.0, and consecutive seeds beginning at 42.


In [ ]:
# train_plsa_restarts is imported from src above.

K = 6
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### PLSA fitting output

The five initializations converged in 40–51 iterations. The run with the highest training log-likelihood (−6,273,009.364) was selected, and the model produced six topic weights for each training and test document. This selection criterion chooses a fit to the training word counts; it does not guarantee the best retrieval ranking.

## 6. Inspect the learned topics and their relation to groups

The next output lists the highest-probability words for each topic and plots training log-likelihood over EM iterations. Topic numbers are arbitrary; interpret a topic from its word distribution rather than its ID. Held-out NMI and ARI compare each test document's dominant topic with its supergroup. They summarize alignment after fitting and do not make PLSA supervised.


In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


test_topic_assignments = np.argmax(theta_test[nonempty_test_mask], axis=1)
test_group_labels = y_test[nonempty_test_mask]
print("Held-out topic/group NMI:", round(normalized_mutual_info_score(test_group_labels, test_topic_assignments), 3))
print("Held-out topic/group ARI:", round(adjusted_rand_score(test_group_labels, test_topic_assignments), 3))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Topic inspection and alignment results

The top words show interpretable structure, but not a clean six-way reproduction of the supergroups. One topic is dominated by software and files, another by drives and hardware terms, and others emphasize religion/worldview, games/sports, or broad political and conversational vocabulary. Several topics contain generic words or mix subject areas, which is consistent with PLSA learning word co-occurrence rather than the declared taxonomy.

The held-out dominant-topic alignment is NMI=0.306 and ARI=0.231. These values indicate partial correspondence, not a one-topic-per-supergroup match. The convergence plot shows the selected model's training likelihood improving until its stopping point; likelihood is a fit diagnostic, not a retrieval metric.

## 7. Controlled topic-boundary probe

To inspect how topic weights respond to mixed content, generate artificial documents from baseball and gun-politics training documents using known token proportions. Identify the topic most associated with each source category from training-document topic averages, infer topic proportions for the mixtures, and summarize 20 sampled mixtures at each proportion. This is a controlled diagnostic of the learned representation, separate from the retrieval benchmark.


In [ ]:
sports_class = source_category_names.index("rec.sport.baseball")
politics_class = source_category_names.index("talk.politics.guns")

source_category_topic_means = np.vstack([
    theta_train[original_y_train == c].mean(axis=0)
    for c in range(len(source_category_names))
])
sports_topic = int(source_category_topic_means[sports_class].argmax())
politics_topic = int(source_category_topic_means[politics_class].argmax())

print("Sports-associated topic:", sports_topic)
print("Politics-associated topic:", politics_topic)
if sports_topic == politics_topic:
    print("Warning: both source categories map to the same dominant topic; interpret estimates cautiously.")


# make_mixed_document is imported from src above.

boundary_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
boundary_rows = []

for ratio in ratios:
    for replicate in range(20):
        mixed_X = make_mixed_document(
            X_train, original_y_train, sports_class, politics_class,
            ratio, boundary_rng, n_tokens=300
        )
        estimate = best_plsa.fold_in(mixed_X, max_iter=100)[0]
        boundary_rows.append({
            "source_category_a": "rec.sport.baseball",
            "source_category_b": "talk.politics.guns",
            "actual_sports_word_ratio": ratio,
            "actual_politics_word_ratio": 1-ratio,
            "estimated_sports_topic": estimate[sports_topic],
            "estimated_politics_topic": estimate[politics_topic],
            "replicate": replicate
        })

boundary_df = pd.DataFrame(boundary_rows)
boundary_summary = boundary_df.groupby("actual_sports_word_ratio").agg(
    estimated_sports_mean=("estimated_sports_topic", "mean"),
    estimated_sports_std=("estimated_sports_topic", "std"),
    estimated_politics_mean=("estimated_politics_topic", "mean"),
    estimated_politics_std=("estimated_politics_topic", "std")
).reset_index()
display(boundary_summary)


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Mixture table results

The estimated mean baseball-associated topic weight rises from 0.079 when the generated documents contain 10% baseball tokens to 0.682 at 90%. The increase is directionally consistent with the known mixture ratio, but it is not a one-to-one estimate: the model underestimates the higher baseball proportions, and the standard deviations show variation among sampled documents. Topic probabilities reflect the model's learned latent representation, not a direct count of source words.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], "k--", label="Ideal reference")
plt.errorbar(
    boundary_summary["actual_sports_word_ratio"],
    boundary_summary["estimated_sports_mean"],
    yerr=boundary_summary["estimated_sports_std"],
    marker="o", capsize=3, label="Estimated sports-topic probability"
)
plt.xlabel("Actual sports word-token proportion")
plt.ylabel("Estimated sports-associated topic probability")
plt.title("PLSA topic-boundary experiment")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. TF-IDF, LSI, and category-structure diagnostic

Fit the retrieval baselines using the same full training split and training-only vocabulary used by PLSA. As a check on the proposed supergroups, compare TF-IDF centroids for the original categories: report average within-supergroup category similarity and similarity to categories outside the group. This diagnostic does not define or optimize the groups and does not use test data; it helps reveal whether the proposed grouping is internally coherent in the training text.

In [ ]:
# All baselines use the shared train-only fitting implementation.
tfidf_vectorizer, lsi_model, tfidf_splits, lsi_splits = fit_baselines(
    docs_train_processed, docs_test_processed, seed=SEED
)
X_train_tfidf, X_test_tfidf = tfidf_splits
lsi_train, lsi_test = lsi_splits



category_centroids = np.vstack([
    np.asarray(X_train_tfidf[original_y_train == category_id].mean(axis=0)).ravel()
    for category_id in range(len(source_category_names))
])
category_centroid_similarity = cosine_similarity(category_centroids)
source_to_group = source_to_supergroup
similarity_rows = []
for group_name, members in SUPERGROUPS.items():
    member_ids = [source_category_names.index(name) for name in members]
    other_ids = [i for i, name in enumerate(source_category_names) if source_to_group[name] != group_name]
    within_values = [
        category_centroid_similarity[i, j]
        for pos, i in enumerate(member_ids)
        for j in member_ids[pos + 1:]
    ]
    between_values = [category_centroid_similarity[i, j] for i in member_ids for j in other_ids]
    similarity_rows.append({
        "supergroup": group_name,
        "source_categories": len(member_ids),
        "mean_within_group_category_cosine": float(np.mean(within_values)) if within_values else np.nan,
        "mean_to_other_group_category_cosine": float(np.mean(between_values)) if between_values else np.nan,
        "closest_external_category": source_category_names[other_ids[int(np.argmax([category_centroid_similarity[member_ids].mean(axis=0)[i] for i in other_ids]))]]
    })
print("Category-centroid similarity check (training data only):")
similarity_df = pd.DataFrame(similarity_rows)
display(similarity_df)


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Baseline and group-structure output

The TF-IDF matrix uses the shared 3,000-term feature limit, and LSI represents documents in 50 dimensions. The centroid table shows that the proposed supergroups are not equally compact: for example, the religion/worldview categories have relatively high within-group similarity, while recreation/consumer exchange is less cohesive. Some groups are also close to categories assigned elsewhere. This supports treating the six-group mapping as a reasoned working taxonomy rather than assuming it forms six perfectly separated clusters.

## 9. Retrieval evaluation

Each supergroup has one fixed broad query. A held-out document is relevant when its original category belongs to the query's supergroup. All methods rank the same non-empty test candidates. The per-query table shows the six query results; the macro table gives each query equal weight.

Precision@10 and Recall@10 count relevant documents in the top ten. AP@10 averages precision at relevant ranks through rank 10, normalized by the smaller of ten and the number of relevant test documents; MAP@10 averages AP@10 over queries. NDCG@10 uses binary supergroup relevance and accounts for ranking positions. Labels supply proxy relevance judgments.


In [ ]:
supergroup_queries = SUPERGROUP_QUERIES

missing = sorted(set(target_names) - set(supergroup_queries))
extra = sorted(set(supergroup_queries) - set(target_names))
print("Supergroups without a query:", missing)
print("Query names not found in supergroups:", extra)
if missing or extra:
    raise ValueError("Provide exactly one query for each evaluation supergroup.")

print("Non-empty test documents:", int(nonempty_test_mask.sum()), "/", len(docs_test_processed))

def evaluate_query(query_text, target_label, k=10):
    processed_query = preprocess_texts([query_text])[0]
    if not processed_query.strip():
        print(f"Warning: query for {target_names[target_label]} became empty after preprocessing.")

    q_tfidf = tfidf_vectorizer.transform([processed_query])
    q_lsi = lsi_model.transform(q_tfidf)
    q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    q_plsa = best_plsa.fold_in(q_counts, max_iter=100)

    sim_tfidf = cosine_similarity(q_tfidf, X_test_tfidf).ravel()[nonempty_test_mask]
    sim_lsi = cosine_similarity(q_lsi, lsi_test).ravel()[nonempty_test_mask]
    sim_plsa = cosine_similarity(q_plsa, theta_test).ravel()[nonempty_test_mask]

    y_eval = y_test[nonempty_test_mask]
    methods = {
        "TF-IDF/VSM": sim_tfidf,
        "LSI": sim_lsi,
        "PLSA": sim_plsa,
    }
    rows = []
    total_relevant = int((y_eval == target_label).sum())
    if total_relevant == 0:
        return rows

    for method_name, scores in methods.items():
        metric = metrics_at_k(scores, y_eval, target_label, k=k, suffix="k")
        rows.append({
            "query_supergroup": target_names[target_label],
            "query_text": query_text,
            "method": method_name,
            "Precision@10": metric["Precision@k"],
            "Recall@10": metric["Recall@k"],
            "MAP@10": metric["AP@k"],
            "NDCG@10": metric["NDCG@k"],
            "relevant_test_documents": total_relevant,
        })
    return rows

eval_rows = []
for category_name in target_names:
    eval_rows.extend(evaluate_query(supergroup_queries[category_name], target_names.index(category_name), k=10))

evaluation_df = pd.DataFrame(eval_rows)
metric_columns = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10"]
print("Per-query results:")
display(evaluation_df[["query_supergroup", "method", *metric_columns, "relevant_test_documents"]].round(4))

mean_metrics = (
    evaluation_df.groupby("method", sort=False)[metric_columns]
    .mean()
    .reset_index()
    .rename(columns={"method": "Method"})
)
print("Macro-average across the six queries:")
display(mean_metrics.round(4))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Retrieval results: single best-restart models

The six-query macro table shows that the broader relevance groups alone do not make a single PLSA model competitive in this run. TF-IDF/VSM leads with Precision@10=0.8833, MAP@10=0.8189, and NDCG@10=0.8982. LSI is next on those measures, with 0.8333, 0.7155, and 0.8077. The best-restart PLSA model scores 0.4333, 0.2858, and 0.4248, respectively. Recall@10 is low for all three because each query has hundreds of relevant test documents while only ten are retrieved.

The per-query table adds an important detail: no method wins every group. TF-IDF is strong for Computing and Recreation/consumer exchange, while LSI reaches perfect top-ten precision and MAP for Digital technology/engineering and Health/space. PLSA's best-restart scores vary considerably across groups and are especially weak for Digital technology/engineering and Politics/international affairs. The plots show these four metrics separately so each scale remains readable; use the tables for exact comparisons.


In [ ]:
metric_columns = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10"]
method_colors = dict(zip(mean_metrics["Method"], plt.cm.tab10.colors))
method_labels = {
    "TF-IDF/VSM": "TF-IDF/VSM",
    "LSI": "LSI",
    "PLSA": "PLSA",
}

for metric in metric_columns:
    values = mean_metrics[metric].to_numpy(dtype=float)
    upper_limit = min(1.15, max(0.05, float(values.max()) * 1.22))
    plot_labels = [method_labels[method] for method in mean_metrics["Method"]]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    bars = ax.bar(
        plot_labels,
        values,
        color=[method_colors[method] for method in mean_metrics["Method"]]
    )
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_ylim(0, upper_limit)
    ax.set_title(f"Macro-average {metric} by method")
    ax.set_ylabel(metric)
    ax.set_xlabel("Retrieval method")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Retrieval plot interpretation

The four charts visualize the macro-average values in the table immediately above. TF-IDF/VSM has the strongest overall single-model scores in this run, with LSI performing well on some individual groups. The PLSA best-restart bars are lower on all four macro metrics. The separate plots make Recall@10 visible without compressing the other measures; the tabular values remain the precise comparison.

## 10. Explore topic count and held-out alignment

The final diagnostic fits one PLSA initialization at each K in 2, 5, 6, 8, 10, and 20, using the same full training matrix. It reports training likelihood and compares dominant held-out topics with the six evaluation groups using NMI and ARI. Because each K uses one initialization and alignment is not retrieval, treat this table as a diagnostic alongside—not a replacement for—the retrieval results above.

In [ ]:
k_rows = []
for k_topics in [2, 5, 6, 8, 10, 20]:
    model = PLSA(
        n_topics=k_topics, max_iter=80, tol=1e-4,
        random_state=SEED, beta=1.0
    ).fit(X_train)
    theta_candidate = model.fold_in(X_test, max_iter=100)
    candidate_topics = np.argmax(theta_candidate[nonempty_test_mask], axis=1)
    heldout_groups = y_test[nonempty_test_mask]
    k_rows.append({
        "K": k_topics,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "heldout_NMI": normalized_mutual_info_score(heldout_groups, candidate_topics),
        "heldout_ARI": adjusted_rand_score(heldout_groups, candidate_topics),
        "test_documents": X_test.shape[0]
    })

k_df = pd.DataFrame(k_rows)
display(k_df)


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### What the K sweep shows

K=8 gives the strongest held-out alignment in this saved sweep (NMI=0.363; ARI=0.343). K=6 is lower (0.306; 0.231), and K=20 does not improve both measures despite its higher training likelihood. The alignment table therefore does not support setting K equal to the number of relevance groups as a general rule; it also does not rank retrieval quality.


## Export this run

Save result tables, convergence, topic words, settings and source hashes. `Result.py` reads these files without repeating model training.

In [ ]:
# This is reached only after all experiment cells complete successfully.
result_directory = export_results(EXPERIMENT, globals())
print("Results saved to:", result_directory)
